# Appendix A4 — LangSmith from Zero

**Who this is for:** a complete beginner who wants to **observe** and **evaluate** LLM apps.
LangSmith answers two questions the other tools don't:

1. **What actually happened?** — **tracing**: every model call, tool call, and chain step, with
   inputs/outputs/latency/cost, in a searchable UI.
2. **Is it any good, and did my change help?** — **evaluation**: run your app over a **dataset**,
   score it with **evaluators**, and compare experiments.

> LangSmith is a hosted service, so uploading traces and running cloud evaluations needs a
> **`LANGSMITH_API_KEY`** (add it as a Colab secret). **Everything conceptual here runs offline:**
> `@traceable` executes without a key (it just doesn't upload), and we run the *evaluation logic*
> locally — which is exactly what the cloud `evaluate()` does. Cloud-only cells are clearly gated.
> Verified against langsmith 0.11.

In [1]:
# === Chapter A4 · standard bootstrap (identical pattern in every notebook) ===
# Runs standalone on a fresh Google Colab VM *or* a local checkout.
import os, sys, subprocess

REPO_URL = "https://github.com/rsalehin/patent-rag-masterclass"
NEED_OCR = False
IN_COLAB = "google.colab" in sys.modules


def _clone_repo(url, target):
    """Clone the repo on Colab. For a PRIVATE repo, authenticate with a GitHub token read from
    Colab Secrets (key 'GITHUB_TOKEN') or the GITHUB_TOKEN env var. The token is never printed."""
    token = None
    try:
        from google.colab import userdata  # type: ignore
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = os.environ.get("GITHUB_TOKEN")
    auth_url = url
    if token and url.startswith("https://github.com/"):
        auth_url = url.replace("https://github.com/", f"https://{token}@github.com/")
    r = subprocess.run(["git", "clone", "--depth", "1", auth_url, target],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)  # avoid leaking the token
    if r.returncode != 0:
        raise RuntimeError(
            "git clone failed. This is a PRIVATE repo, so Colab needs a GitHub token:\n"
            "  1) Create a token (scope: repo) at https://github.com/settings/tokens\n"
            "  2) In Colab, open the key icon (Secrets) in the left sidebar, add a secret named\n"
            "     GITHUB_TOKEN, paste the token, and enable 'Notebook access'.\n"
            "  3) Re-run this cell.\n"
            "  (Alternatively, make the GitHub repo public — then no token is needed.)")


if IN_COLAB:
    target = "/content/patent-rag-masterclass"
    if not os.path.isdir(target):
        if not REPO_URL:
            raise RuntimeError("Set REPO_URL to this repo's GitHub URL (see README.md).")
        _clone_repo(REPO_URL, target)
    os.chdir(target)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    if NEED_OCR:
        subprocess.run(["apt-get", "install", "-y", "-q", "tesseract-ocr"], check=False)

# Ensure the repo root (containing patentrag/) is importable.
for _cand in [os.getcwd()] + [os.path.dirname(os.getcwd())]:
    if os.path.isdir(os.path.join(_cand, "patentrag")):
        if _cand not in sys.path:
            sys.path.insert(0, _cand)
        break

from patentrag import bootstrap as bs
bs.setup_environment(REPO_URL, need_ocr=NEED_OCR)
bs.set_seeds()
_env = bs.environment_report()
print("Chapter A4 bootstrap OK")
print("  Python", _env["python"], "| Colab:", _env["in_colab"], "| CPU cores:", _env["cpu_count"])
print("  torch", _env["torch"], "| CUDA:", _env["cuda_available"], "| tesseract:", _env["tesseract"])

Chapter A4 bootstrap OK
  Python 3.12.10 | Colab: False | CPU cores: 24
  torch 2.12.0.dev20260304+cu130 | CUDA: True | tesseract: True


In [2]:
# Install the LangChain / LangGraph / LangSmith stack (extra deps for the appendices).
# No-op locally if already installed; installs on a fresh Colab VM.
import sys, subprocess, os
if os.path.exists("requirements-appendix.txt"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-appendix.txt"], check=True)
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "langchain==1.3.18", "langchain-core==1.6.1", "langchain-text-splitters==1.1.2",
                    "langgraph==1.2.11", "langsmith==0.11.2", "langchain-openai==1.6.0"], check=True)
print("LangChain stack ready.")

LangChain stack ready.


In [3]:
# --- optional real LLM + a deterministic offline fallback -------------------------------------
# Everything in these appendices runs with NO API key using deterministic fake models. To use a
# REAL model, add a Colab Secret (key icon, left sidebar) named LLM_API_KEY (any OpenAI-compatible
# endpoint; optionally LLM_BASE_URL and LLM_MODEL), or OPENAI_API_KEY. For LangSmith tracing add
# LANGSMITH_API_KEY. Secrets are pulled into environment variables here; nothing is printed.
import os
def _load_secret(name):
    try:
        from google.colab import userdata  # type: ignore
        v = userdata.get(name)
        if v:
            os.environ[name] = v
    except Exception:
        pass
for _n in ["OPENAI_API_KEY", "LLM_API_KEY", "LLM_BASE_URL", "LLM_MODEL", "LANGSMITH_API_KEY"]:
    _load_secret(_n)

LIVE_LLM = bool(os.environ.get("LLM_API_KEY") or os.environ.get("OPENAI_API_KEY"))

def get_chat_model(fake_responses=None, temperature: float = 0.0):
    """Return a real ChatOpenAI if a key is configured, else a deterministic fake chat model."""
    if LIVE_LLM:
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=os.environ.get("LLM_MODEL", "gpt-4o-mini"),
                          base_url=os.environ.get("LLM_BASE_URL"),
                          api_key=os.environ.get("LLM_API_KEY") or os.environ.get("OPENAI_API_KEY"),
                          temperature=temperature)
    from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
    return GenericFakeChatModel(messages=iter(fake_responses or ["(deterministic fake-model answer)"]))

# A scripted tool-calling model so the REAL agent APIs can run offline (it replays AIMessages,
# including tool_calls, and implements bind_tools so agents accept it).
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.outputs import ChatResult, ChatGeneration
from pydantic import PrivateAttr
class ScriptedChatModel(BaseChatModel):
    responses: list
    _i: int = PrivateAttr(default=0)
    def _generate(self, messages, stop=None, run_manager=None, **kw):
        msg = self.responses[min(self._i, len(self.responses) - 1)]
        self._i += 1
        return ChatResult(generations=[ChatGeneration(message=msg)])
    def bind_tools(self, tools, **kw):
        return self
    @property
    def _llm_type(self):
        return "scripted"

print("LLM helpers ready. Live model configured:", LIVE_LLM)

LLM helpers ready. Live model configured: False


In [4]:
# Is LangSmith cloud configured? (add LANGSMITH_API_KEY as a Colab secret to enable uploads/eval)
import os
LANGSMITH = bool(os.environ.get("LANGSMITH_API_KEY"))
print("LangSmith cloud configured:", LANGSMITH)
if LANGSMITH:
    os.environ["LANGSMITH_TRACING"] = "true"       # turn on auto-tracing for LangChain runnables
    os.environ.setdefault("LANGSMITH_PROJECT", "patent-rag-appendix")

LangSmith cloud configured: False


## 1. Tracing with `@traceable`

Decorate any function with **`@traceable`** and LangSmith records it as a **run**. Nested
`@traceable` calls form a **run tree** (parent → children) — so you can see a whole request's
breakdown. Without a key the functions run normally; with `LANGSMITH_TRACING=true` + a key, each
run is uploaded to the UI.

In [5]:
from langsmith import traceable
from langsmith.run_helpers import get_current_run_tree

@traceable(run_type="tool", name="retrieve")
def retrieve(query: str):
    return ["doc about HNSW", "doc about quantization"]

@traceable(run_type="llm", name="generate")
def generate(query: str, docs: list):
    return f"Based on {len(docs)} docs: HNSW is a graph index for ANN search."

@traceable(run_type="chain", name="rag_pipeline")
def rag(query: str):
    docs = retrieve(query)          # becomes a child run
    answer = generate(query, docs)  # becomes a sibling child run
    tree = get_current_run_tree()
    return {"answer": answer, "traced_as": tree.name if tree else "(tracing off — offline)"}

print(rag("What is HNSW?"))

{'answer': 'Based on 2 docs: HNSW is a graph index for ANN search.', 'traced_as': '(tracing off — offline)'}


## 2. Turning tracing on

Tracing is controlled by environment variables — no code changes to your app:

```python
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = "ls-..."      # your key (use a Colab secret)
os.environ["LANGSMITH_PROJECT"] = "my-project"  # groups runs in the UI
```

Once set, **every LangChain/LangGraph runnable auto-traces** (no decorators needed), and your
`@traceable` functions upload too. We enable it automatically above if the secret is present.

In [6]:
# LangChain runnables auto-trace when LANGSMITH_TRACING=true. This runs either way:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
chain = ChatPromptTemplate.from_template("Define {term} in one line.") | get_chat_model(["A concise definition."]) | StrOutputParser()
print("chain output:", chain.invoke({"term": "BM25"}))
print("(with LANGSMITH_TRACING=true this call appears as a trace in the LangSmith UI)")

chain output: A concise definition.
(with LANGSMITH_TRACING=true this call appears as a trace in the LangSmith UI)


## 3. Metadata & tags

Attach **tags** and **metadata** to runs so you can filter them in the UI (by version, user,
experiment, etc.). You pass them via the runnable config.

In [7]:
tagged_chain = (ChatPromptTemplate.from_template("Define {term} in one line.")
                | get_chat_model(["A concise definition."]) | StrOutputParser())
result = tagged_chain.invoke({"term": "NDCG"},
                             config={"tags": ["appendix", "demo"], "metadata": {"version": "v1", "user": "ada"}})
print("output:", result)
print("tags/metadata travel with the trace (visible + filterable in the UI).")

output: A concise definition.
tags/metadata travel with the trace (visible + filterable in the UI).


## 4. Datasets — the ground truth for evaluation

A **dataset** is a named collection of **examples**, each with **inputs** and (usually)
**reference outputs** (the expected answer). You evaluate an app *against* a dataset. In the cloud
you create datasets with the `Client`; offline we define the examples as a plain list — the shape
is identical.

In [8]:
dataset = [
    {"inputs": {"question": "capital of France"},   "reference": {"answer": "Paris"}},
    {"inputs": {"question": "capital of Japan"},    "reference": {"answer": "Tokyo"}},
    {"inputs": {"question": "capital of Germany"},  "reference": {"answer": "Berlin"}},
]
print(f"{len(dataset)} examples; first:", dataset[0])

# Cloud equivalent (needs a key): create a dataset you can reuse across experiments.
if LANGSMITH:
    from langsmith import Client
    client = Client()
    ds = client.create_dataset("capitals-demo")
    client.create_examples(dataset=ds.id,
                            inputs=[e["inputs"] for e in dataset],
                            outputs=[e["reference"] for e in dataset])
    print("uploaded dataset to LangSmith:", ds.name)
else:
    print("SKIPPED cloud upload (no LANGSMITH_API_KEY) — using the local list above.")

3 examples; first: {'inputs': {'question': 'capital of France'}, 'reference': {'answer': 'Paris'}}
SKIPPED cloud upload (no LANGSMITH_API_KEY) — using the local list above.


## 5. Evaluators — functions that score a prediction

An **evaluator** compares your app's **output** to the **reference** and returns a score.
LangSmith evaluators are just functions returning `{"key": ..., "score": ...}`. Three common
kinds:

- **exact / heuristic** — deterministic string checks.
- **similarity** — soft overlap when wording varies.
- **LLM-as-judge** — an LLM rates the answer against a rubric (powerful but needs a model; beware
  judge bias — see main-series Chapter 10).

In [9]:
def exact_match(output: str, reference: str) -> dict:
    return {"key": "exact_match", "score": float(output.strip().lower() == reference.strip().lower())}

def contains_answer(output: str, reference: str) -> dict:
    return {"key": "contains", "score": float(reference.strip().lower() in output.strip().lower())}

def token_overlap(output: str, reference: str) -> dict:
    a, b = set(output.lower().split()), set(reference.lower().split())
    return {"key": "token_overlap", "score": len(a & b) / len(a | b) if (a | b) else 0.0}

print(exact_match("Paris", "paris"))
print(contains_answer("The capital is Paris.", "Paris"))
print(token_overlap("the capital is paris", "paris capital"))

{'key': 'exact_match', 'score': 1.0}
{'key': 'contains', 'score': 1.0}
{'key': 'token_overlap', 'score': 0.5}


### An LLM-as-judge evaluator

A judge LLM scores the answer against a rubric. Offline we use the fake model returning a canned
score; with a real model (Colab secret) it genuinely judges. Always keep deterministic evaluators
alongside a judge — never trust the judge alone.

In [10]:
import json, re
def llm_judge(question: str, output: str, reference: str) -> dict:
    judge = get_chat_model(['{"score": 1, "reason": "matches the reference answer"}'])
    prompt = (f"Question: {question}\nReference: {reference}\nAnswer: {output}\n"
              'Score 1 if the answer is correct, else 0. Reply as JSON {"score": int, "reason": str}.')
    raw = judge.invoke(prompt).content
    try:
        data = json.loads(re.search(r"\{.*\}", raw, re.DOTALL).group(0))
    except Exception:
        data = {"score": 0, "reason": "unparseable"}
    return {"key": "llm_judge", "score": float(data["score"]), "comment": data.get("reason", "")}

print(llm_judge("capital of France", "Paris", "Paris"))

{'key': 'llm_judge', 'score': 1.0, 'comment': 'matches the reference answer'}


## 6. Run an evaluation (offline) — what `evaluate()` does

Evaluation is a loop: **for each example → run the app on its inputs → score the output with every
evaluator → aggregate**. Here it is by hand, so the machinery is clear. Our "app" (the `target`)
is a tiny function; swap in any chain/agent.

In [11]:
import pandas as pd
# the app under test (deterministic fake "knows" the capitals)
CAPITALS = {"France": "Paris", "Japan": "Tokyo", "Germany": "Berlin"}
def target(inputs: dict) -> dict:
    q = inputs["question"]
    country = q.split()[-1].title()
    return {"answer": CAPITALS.get(country, "unknown")}

evaluators = [exact_match, contains_answer, token_overlap]
rows = []
for ex in dataset:
    out = target(ex["inputs"])["answer"]
    ref = ex["reference"]["answer"]
    scores = {ev(out, ref)["key"]: ev(out, ref)["score"] for ev in evaluators}
    scores["judge"] = llm_judge(ex["inputs"]["question"], out, ref)["score"]
    rows.append({"question": ex["inputs"]["question"], "output": out, "reference": ref, **scores})
results = pd.DataFrame(rows)
print("per-example scores:")
print(results.to_string(index=False))
print("\naggregate:", {c: round(results[c].mean(), 2) for c in ["exact_match", "contains", "token_overlap", "judge"]})

per-example scores:
          question output reference  exact_match  contains  token_overlap  judge
 capital of France  Paris     Paris          1.0       1.0            1.0    1.0
  capital of Japan  Tokyo     Tokyo          1.0       1.0            1.0    1.0
capital of Germany Berlin    Berlin          1.0       1.0            1.0    1.0

aggregate: {'exact_match': np.float64(1.0), 'contains': np.float64(1.0), 'token_overlap': np.float64(1.0), 'judge': np.float64(1.0)}


## 7. The cloud `evaluate()` API — the same thing, hosted

On the platform you call **`evaluate(target, data=..., evaluators=...)`**: it runs the target over
the dataset, applies the evaluators, and records an **experiment** you can compare against others
in the UI. It needs a key, so we gate it.

In [12]:
if LANGSMITH:
    from langsmith import evaluate
    def ls_exact(outputs: dict, reference_outputs: dict) -> dict:
        return {"key": "exact_match",
                "score": float(outputs["answer"].strip().lower() == reference_outputs["answer"].strip().lower())}
    experiment = evaluate(target, data="capitals-demo", evaluators=[ls_exact],
                          experiment_prefix="capitals-baseline")
    print("ran a LangSmith experiment:", experiment)
else:
    print("SKIPPED cloud evaluate() (no LANGSMITH_API_KEY).")
    print("It performs exactly the loop in section 6 — running the target over the dataset,")
    print("applying evaluators — but hosted, with experiment tracking + comparison in the UI.")
    print("To enable: add LANGSMITH_API_KEY (and optionally an LLM key) as Colab secrets.")

SKIPPED cloud evaluate() (no LANGSMITH_API_KEY).
It performs exactly the loop in section 6 — running the target over the dataset,
applying evaluators — but hosted, with experiment tracking + comparison in the UI.
To enable: add LANGSMITH_API_KEY (and optionally an LLM key) as Colab secrets.


## 8. The LangSmith workflow

The loop teams actually run:

```
build app  →  trace it (see real behavior)  →  turn interesting/failing traces into dataset examples
        →  write evaluators  →  evaluate()  →  compare experiments  →  ship the version that scored best
        →  collect production feedback  →  grow the dataset  →  repeat
```

**Feedback** (👍/👎, corrections, human scores) attaches to runs and can seed new dataset examples,
closing the loop between production and evaluation.

## 9. Tie-in: evaluate a tiny patent-QA target

The same offline evaluation over patent questions, mixing an exact-ish evaluator with the judge.

In [13]:
import json
CORPUS = {json.loads(p.read_text(encoding='utf-8'))["publication_number"]:
          json.loads(p.read_text(encoding='utf-8'))["title"]
          for p in sorted((bs.DATA/'corpus').glob('US*.json'))}
patent_ds = [
    {"inputs": {"pub": "US9081550B2"}, "reference": {"answer": CORPUS["US9081550B2"]}},
    {"inputs": {"pub": "US8930304B2"}, "reference": {"answer": CORPUS["US8930304B2"]}},
]
def title_lookup(inputs): return {"answer": CORPUS.get(inputs["pub"], "unknown")}
patent_rows = []
for ex in patent_ds:
    out = title_lookup(ex["inputs"])["answer"]; ref = ex["reference"]["answer"]
    patent_rows.append({"pub": ex["inputs"]["pub"], "exact": exact_match(out, ref)["score"],
                        "overlap": round(token_overlap(out, ref)["score"], 2)})
patent_eval = pd.DataFrame(patent_rows)
print(patent_eval.to_string(index=False))

        pub  exact  overlap
US9081550B2    1.0      1.0
US8930304B2    1.0      1.0


### You now know LangSmith

**tracing** (`@traceable`, run trees, auto-tracing via env vars) · **tags & metadata** ·
**datasets** (inputs + reference outputs) · **evaluators** (heuristic, similarity, LLM-as-judge) ·
the **evaluation loop** (and the hosted `evaluate()`) · the **dev → trace → dataset → evaluate →
compare** workflow.

Together, the three appendices cover the modern LLM-app stack: **LangChain** (compose), **LangGraph**
(orchestrate stateful agents), **LangSmith** (observe + evaluate).

## Chapter invariants

In [14]:
assert rag("q")["answer"]                                        # traceable pipeline runs offline
assert exact_match("Paris", "paris")["score"] == 1.0             # evaluator correctness
assert token_overlap("a b", "b c")["score"] == 1/3
assert set(results["exact_match"]) == {1.0}                      # target got every capital right
assert results["judge"].mean() >= 0                              # judge produced scores
assert len(patent_eval) == 2 and (patent_eval["exact"] == 1.0).all()   # patent-QA exact matches
print("All Appendix A4 invariants hold.")

All Appendix A4 invariants hold.


In [15]:
# === Chapter A4 validation footer ===
import time, platform, sys, importlib.metadata as _md
_pkgs = ['langsmith', 'langchain', 'langchain-core']
print("Chapter A4 — environment")
print("  Python :", sys.version.split()[0], "on", platform.system(), platform.release())
for _p in _pkgs:
    try: print(f"  {_p:24}: {_md.version(_p)}")
    except Exception: print(f"  {_p:24}: (not installed)")
print()
print("CHAPTER A4 VALIDATION: PASS")

Chapter A4 — environment
  Python : 3.12.10 on Windows 11
  langsmith               : 0.11.2
  langchain               : 1.3.18
  langchain-core          : 1.6.1

CHAPTER A4 VALIDATION: PASS
